# 🧠 Notebook 05 — Explainable AI & Ethics
**CCS3440 Artificial Intelligence | SmartCare AI Risk Prediction**

---
### 🎯 Objectives (Tasks 07 & 08)
- **Task 07**: Apply **SHAP** (SHapley Additive exPlanations) to explain model predictions
  - Global feature importance (summary plot)
  - Local explanation for individual patients (waterfall / force plots)
- **Task 08**: **Ethical Analysis** — Bias, fairness, privacy, and accountability

---

## 1️⃣ Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import joblib
import warnings
warnings.filterwarnings('ignore')

shap.initjs()
print('✅ SHAP and libraries imported')

## 2️⃣ Load Data & Best Model

In [ ]:
import json

X_test = pd.read_csv('../data/processed/X_test.csv')
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

# Load the best model (change name if needed)
with open('../models/final_model_selection.json') as f:
    selection = json.load(f)

best_model_name = selection.get('best_model', 'random_forest')
model = joblib.load(f'../models/{best_model_name}.pkl')

print(f'✅ Loaded best model: {best_model_name}')

---
## 🔍 Task 07 — SHAP Explainability

### 3️⃣ Compute SHAP Values

In [ ]:
# Use TreeExplainer for tree-based models (RF, XGBoost, DT)
# Use KernelExplainer for Logistic Regression
try:
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    print('✅ TreeExplainer used')
except Exception:
    explainer = shap.KernelExplainer(model.predict_proba, shap.sample(X_test, 100))
    shap_values = explainer.shap_values(X_test)
    print('✅ KernelExplainer used')

### 4️⃣ Global Feature Importance — SHAP Summary Plot

In [ ]:
plt.figure(figsize=(10, 6))

# For binary classification with TreeExplainer, shap_values may be a list
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

shap.summary_plot(sv, X_test, plot_type='bar', show=False)
plt.title('SHAP Feature Importance (Global)', fontsize=14)
plt.tight_layout()
plt.savefig('../reports/figures/shap_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ SHAP summary plot saved')

### 5️⃣ SHAP Beeswarm Plot

In [ ]:
plt.figure(figsize=(10, 7))
shap.summary_plot(sv, X_test, show=False)
plt.title('SHAP Beeswarm Plot — Feature Impact Distribution', fontsize=14)
plt.tight_layout()
plt.savefig('../reports/figures/shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ SHAP beeswarm plot saved')

### 6️⃣ Local Explanation — Individual Patient (Waterfall Plot)

In [ ]:
# Explain prediction for the first patient in test set
patient_idx = 0

shap.waterfall_plot(
    shap.Explanation(
        values=sv[patient_idx],
        base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
        data=X_test.iloc[patient_idx].values,
        feature_names=X_test.columns.tolist()
    ),
    show=False
)
plt.title(f'SHAP Waterfall — Patient #{patient_idx}', fontsize=13)
plt.tight_layout()
plt.savefig('../reports/figures/shap_waterfall_patient0.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Waterfall plot saved')

---
## ⚖️ Task 08 — AI Ethics Analysis

This section discusses the ethical considerations of deploying AI for medical risk prediction.

### 7️⃣ Bias & Fairness Analysis

**Potential Sources of Bias:**
- **Selection Bias**: The dataset may not represent all demographic groups equally (e.g., age, ethnicity).
- **Historical Bias**: If historical medical decisions were biased, the model may learn and perpetuate those patterns.
- **Label Bias**: Diagnostic labels in training data may reflect access-to-care disparities.

**Mitigation Strategies:**
- Evaluate model performance across demographic subgroups.
- Use fairness-aware training techniques.
- Conduct regular audits of model predictions.

In [ ]:
# TODO: Subgroup fairness analysis
# Example: Evaluate ROC-AUC separately for different age groups
# X_test_with_labels = X_test.copy()
# X_test_with_labels['y_true'] = y_test.values
# X_test_with_labels['y_pred'] = model.predict(X_test)
# age_groups = pd.cut(X_test_with_labels['Age'], bins=[0, 30, 50, 120], labels=['Young', 'Middle', 'Senior'])
# for group in age_groups.unique():
#     subset = X_test_with_labels[age_groups == group]
#     print(f'{group}: Accuracy = {accuracy_score(subset["y_true"], subset["y_pred"]):.4f}')

print('⚠️ TODO: Complete subgroup fairness analysis above')

### 8️⃣ Transparency & Accountability

| Principle | Implementation in SmartCare AI |
|---|---|
| **Transparency** | SHAP explanations provided for every prediction |
| **Accountability** | Model versioning and selection metadata saved |
| **Non-maleficence** | Model is decision-support only, not a replacement for clinicians |
| **Privacy** | No personal identifiers used in model training |
| **Beneficence** | Designed to improve early disease detection |

### 9️⃣ Recommendations
1. Always require a clinician to review AI predictions before any medical decision.
2. Retrain and audit the model periodically as new patient data becomes available.
3. Ensure informed consent for any patient data used in model training.
4. Document model limitations prominently in the clinical interface.